## Evaluation metrics

In the binary classification module we trained a logistic regression model that predicts customer churn . But how do we know if it's good , accurate and correct ?

We will use metrics to evaluate the model 

A metric is a function that compares the predictions with the actual values and outputs a single number that tells how good the predictions are.

That is all a metric is: predictions go in, the true values go in, and one number comes out. Different metrics look at different aspects of the predictions, and picking the right one for the problem is the topic of this whole module.






In [14]:
import pandas as pd # To import pandas library for data manipulation and analysis
import numpy as np # To import numpy library for numerical computations

import matplotlib.pyplot as plt # To import matplotlib library for data visualization
import seaborn as sns # To import seaborn library for statistical data visualization

# To display plots inline in Jupyter Notebook
%matplotlib inline 

In [15]:
from sklearn.model_selection import train_test_split # To import the train_test_split function from scikit-learn for splitting the dataset into training and testing sets
from sklearn.feature_extraction import DictVectorizer # To import the DictVectorizer class from scikit-learn for converting feature dictionaries into numerical feature vectors
from sklearn.linear_model import LogisticRegression # To import the LogisticRegression class from scikit-learn for building a logistic regression model

We will use the same dataset as in the churn model for the telcom company 
Dataset used in this project is donwloded from Kaggle , URL: https://www.kaggle.com/datasets/blastchar/telco-customer-churn

In [16]:
df = pd.read_csv('/workspaces/ML-Projects/Classification/WA_Fn-UseC_-Telco-Customer-Churn.csv') # To read the CSV file containing customer churn data into a pandas DataFrame
df

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7038,6840-RESVB,Male,0,Yes,Yes,24,Yes,Yes,DSL,Yes,...,Yes,Yes,Yes,Yes,One year,Yes,Mailed check,84.80,1990.5,No
7039,2234-XADUH,Female,0,Yes,Yes,72,Yes,Yes,Fiber optic,No,...,Yes,No,Yes,Yes,One year,Yes,Credit card (automatic),103.20,7362.9,No
7040,4801-JZAZL,Female,0,Yes,Yes,11,No,No phone service,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.60,346.45,No
7041,8361-LTMKD,Male,1,Yes,No,4,Yes,Yes,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Mailed check,74.40,306.6,Yes


We Will do the necessary data preparation and setting up the validation framework


In [17]:
df.columns = df.columns.str.lower().str.replace(' ', '_') # To convert all column names to lowercase and replace spaces with underscores for consistency and easier access

categorical_columns = list(df.dtypes[df.dtypes == 'str'].index) # To create a list of column names that have data type 'object', which typically indicates categorical variables

for col in categorical_columns: # To iterate over each column name in the list of categorical columns
    df[col] = df[col].str.lower().str.replace(' ', '_') # To convert all string values in the column to lowercase and replace spaces with underscores for consistency and easier analysis
    
df

,customerid,gender,seniorcitizen,partner,dependents,tenure,phoneservice,multiplelines,internetservice,onlinesecurity,...,deviceprotection,techsupport,streamingtv,streamingmovies,contract,paperlessbilling,paymentmethod,monthlycharges,totalcharges,churn
0,7590-vhveg,female,0,yes,no,1,no,no_phone_service,dsl,no,...,no,no,no,no,month-to-month,yes,electronic_check,29.85,29.85,no
1,5575-gnvde,male,0,no,no,34,yes,no,dsl,yes,...,yes,no,no,no,one_year,no,mailed_check,56.95,1889.5,no
2,3668-qpybk,male,0,no,no,2,yes,no,dsl,yes,...,no,no,no,no,month-to-month,yes,mailed_check,53.85,108.15,yes
3,7795-cfocw,male,0,no,no,45,no,no_phone_service,dsl,yes,...,yes,yes,no,no,one_year,no,bank_transfer_(automatic),42.30,1840.75,no
4,9237-hqitu,female,0,no,no,2,yes,no,fiber_optic,no,...,no,no,no,no,month-to-month,yes,electronic_check,70.70,151.65,yes
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7038,6840-resvb,male,0,yes,yes,24,yes,yes,dsl,yes,...,yes,yes,yes,yes,one_year,yes,mailed_check,84.80,1990.5,no
7039,2234-xaduh,female,0,yes,yes,72,yes,yes,fiber_optic,no,...,yes,no,yes,yes,one_year,yes,credit_card_(automatic),103.20,7362.9,no
7040,4801-jzazl,female,0,yes,yes,11,no,no_phone_service,dsl,yes,...,no,no,no,no,month-to-month,yes,electronic_check,29.60,346.45,no
7041,8361-ltmkd,male,1,yes,no,4,yes,yes,fiber_optic,no,...,no,no,no,no,month-to-month,yes,mailed_check,74.40,306.6,yes


In [18]:
tc = pd.to_numeric(df.totalcharges, errors='coerce') # To convert the 'totalcharges' column to numeric type, coercing any errors (e.g., non-numeric values) to NaN

df.totalcharges = df.totalcharges.fillna(0) # To fill any NaN values in the 'totalcharges' column with 0, ensuring that there are no missing values in this numeric column

df.churn = (df.churn == 'yes').astype(int) # To convert the 'churn' column to a binary format where 'yes' is represented as 1 and 'no' as 0, making it suitable for machine learning models that require numerical input.



In [19]:
df_full_train , df_test = train_test_split(df, test_size = 0.2 , random_state = 1) # To split the DataFrame 'df' into training and testing sets, with 20% of the data allocated for testing and a random state of 1 for reproducibility , so wehen execute the code you will get the same result. This function returns four outputs: X_train, X_test, y_train, y_test.
df_train , df_val = train_test_split(df_full_train, test_size = 0.25 , random_state = 1) # To further split the training set into a smaller training set and a validation set, with 25% of the data allocated for validation. This allows for model evaluation during training.

df_train = df_train.reset_index(drop = True )
df_val = df_val.reset_index(drop = True ) 
df_test = df_test.reset_index(drop = True ) # To reset the index of the training, validation, and testing DataFrames, dropping the old index. This is useful for ensuring that the indices are sequential and start from 0, which can help avoid confusion during analysis and modeling.

y_train = df_train.churn.values # To extract the target variable 'churn' from the training set and convert it to a NumPy array for model training
y_val = df_val.churn.values # To extract the target variable 'churn' from the validation set and convert it to a NumPy array for model evaluation
y_test = df_test.churn.values # To extract the target variable 'churn' from the testing set and convert it to a NumPy array for final model evaluation

del df_train['churn'] # To remove the 'churn' column from the training DataFrame, as it has already been extracted into the y_train array and is no longer needed in the feature set.
del df_val['churn'] # To remove the 'churn' column from the validation DataFrame, as it has already been extracted into the y_val array and is no longer needed in the feature set.
del df_test['churn'] # To remove the 'churn' column from the testing DataFrame, as it has already been extracted into the y_test array and is no



## Accuracy

